In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_IHBAS, Dilshad Garden, Delhi - CPCB.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,BP,Xylene,AT,RF
0,01-01-2025 00:00,02-01-2025 00:00,60.29,191.66,16.30,34.47,31.56,47.68,12.03,0.8,...,2.89,1.32,1.79,81.72,1.07,98.47,NaN,1.88,NaN,NaN
1,02-01-2025 00:00,03-01-2025 00:00,61.47,183.32,15.69,32.67,30.13,47.57,8.86,1.6,...,3.02,1.13,1.55,85.21,1.15,84.99,NaN,1.62,NaN,NaN
2,03-01-2025 00:00,04-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04-01-2025 00:00,05-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05-01-2025 00:00,06-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,12-11-2025 00:00,13-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
641,13-11-2025 00:00,14-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
642,14-11-2025 00:00,15-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
643,15-11-2025 00:00,16-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (645, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['BP']
Dropped rows (>70% NaN): 329
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
NH3            0
SO2            0
CO             0
Ozone          0
Benzene        0
Toluene        0
Eth-Benzene    0
MP-Xylene      0
RH             0
WS             0
WD             0
Xylene         0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (316, 19)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   60.29  191.66  16.30  34.47  31.56   
1  02-01-2025 00:00  03-01-2025 00:00   61.47  183.32  15.69  32.67  30.13   
2  07-01-2025 00:00  08-01-2025 00:00      79  180.20  18.77  35.33  33.42   
3  08-01-2025 00:00  09-01-2025 00:00   83.55  175.73  14.67  34.55  30.30   
4  09-01-2025 00:00  10-01-2025 00:00  122.42  296.04  15.11  35.58  31.21   

     NH3    SO2    CO  Ozone  Benzene  Toluene  Eth-Benzene  MP-Xylene     RH  \
0  47.68  12.03  0.80   9.06     1.98     2.89         1.32       1.79  81.72   
1  47.57   8.86  1.60  12.56     1.98     3.02         1.13       1.55  85.21   
2  48.15  16.48  1.41  11.35     2.98     1.60         1.12       1.84  78.50   
3  47.70  12.32  1.68  11.49     2.15     2.75         1.34       1.81  81.11   
4  47.82  20.26  1.19  19.71     2.39     3.55         1.48       1.72  83.75   

     WS      WD  Xyle

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,Xylene
0,01-01-2025 00:00,02-01-2025 00:00,60.29,0.966193,0.630908,-0.466598,0.283572,1.468387,0.012076,-0.938033,-1.682604,-0.450690,-0.441663,-0.368190,0.171754,1.187919,0.186473,-0.933700,0.617136
1,02-01-2025 00:00,03-01-2025 00:00,61.47,0.831746,0.516419,-0.622033,-0.037586,1.460111,-0.584729,0.875201,-1.182343,-0.450690,-0.348919,-0.729814,-0.181441,1.363441,0.533859,-1.140251,0.186274
2,07-01-2025 00:00,08-01-2025 00:00,79,0.781450,1.094495,-0.392335,0.701303,1.503746,0.849864,0.444558,-1.355290,0.868362,-1.361968,-0.748847,0.245336,1.025975,1.358901,-1.475667,0.103415
3,08-01-2025 00:00,09-01-2025 00:00,83.55,0.709390,0.324978,-0.459690,0.000593,1.469891,0.066674,1.056525,-1.335280,-0.226451,-0.541541,-0.330124,0.201187,1.157240,-0.768839,-0.656510,0.600565
4,09-01-2025 00:00,10-01-2025 00:00,122.42,2.648872,0.407560,-0.370747,0.204967,1.478919,1.561513,-0.054081,-0.160382,0.090121,0.029191,-0.063664,0.068739,1.290013,-0.985956,-0.371966,-0.402020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,12-11-2025 00:00,13-11-2025 00:00,66.65,-0.037482,-1.349191,0.225948,-0.679903,-0.554593,2.094308,-0.190074,0.765815,-0.239642,-0.477333,0.316994,0.024589,-0.724721,-0.812263,-1.480877,0.501135
312,13-11-2025 00:00,14-11-2025 00:00,88.51,-0.037482,-1.321038,0.223358,-0.657445,-0.525253,1.906041,1.464503,2.353785,-0.213261,-0.513004,0.488289,0.083455,-0.846430,-1.029379,-1.478578,0.351990
313,14-11-2025 00:00,15-11-2025 00:00,164.69,-0.037482,-1.390482,0.236310,-0.702362,-0.545565,1.986996,0.535220,2.368078,-0.200070,-0.455931,0.336027,-0.122575,-0.669399,-0.899109,-1.347875,0.633708
314,15-11-2025 00:00,16-11-2025 00:00,137.05,-0.037482,-1.343560,0.238038,-0.675412,-0.545565,2.391769,0.625882,1.214620,-0.200070,-0.441663,0.431191,-0.004843,-0.435034,-1.116225,-1.082790,0.484563


In [10]:
df.to_excel('IHBS2025.xlsx', index=False)